Extracting Embeddings(When submitted this is one file combined with the other two)

In [ ]:
# --- Extract image embeddings for all datasets ---
train_img_embeddings = get_image_embeddings_batchwise(train_dataset, imagemodel, batch_size=16)
val_img_embeddings   = get_image_embeddings_batchwise(val_dataset, imagemodel, batch_size=16)
test_img_embeddings  = get_image_embeddings_batchwise(test_dataset, imagemodel, batch_size=16)

# --- Extract text embeddings ---
# Load trained model first
text_model = TextClassifier(num_classes=4, base_model=base_model).to(device)
text_model.load_state_dict(torch.load("/home/le.song/Assignment2/clean_text_classifier.pth"))

train_text_embeddings = extract_text_embeddings_batchwise(train_texts, text_model, tokenizer, batch_size=16)
val_text_embeddings   = extract_text_embeddings_batchwise(val_texts, text_model, tokenizer, batch_size=16)
test_text_embeddings  = extract_text_embeddings_batchwise(test_texts, text_model, tokenizer, batch_size=16)

Define Fusion Dataset

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, img_emb, text_emb, labels):
        self.img_emb = torch.tensor(img_emb, dtype=torch.float32)
        self.text_emb = torch.tensor(text_emb, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "img_emb": self.img_emb[idx],
            "text_emb": self.text_emb[idx],
            "label": self.labels[idx]
        }

train_fusion = FusionDataset(train_img_embeddings, train_text_embeddings, train_labels)
val_fusion   = FusionDataset(val_img_embeddings, val_text_embeddings, val_labels)
test_fusion  = FusionDataset(test_img_embeddings, test_text_embeddings, test_labels)

train_loader = DataLoader(train_fusion, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_fusion, batch_size=16)
test_loader  = DataLoader(test_fusion, batch_size=16)

In [ ]:
Define Fusion Classifier

In [ ]:
fusion_input_dim = train_img_embeddings.shape[1] + train_text_embeddings.shape[1]

class FusionClassifier(nn.Module):
    def __init__(self, input_dim=fusion_input_dim, hidden_dim=128, num_classes=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, img_emb, text_emb):
        x = torch.cat([img_emb, text_emb], dim=1)
        return self.fc(x)

fusion_model = FusionClassifier().to(device)

Training Loop with freeze → unfreeze schedule (Same timed-unfreeze mechanism as the image model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-3)

FROZEN_EPOCHS = 10  # train classifier head only
FINE_TUNE_EPOCHS = 5 # unfreeze embeddings and fine-tune

# Freeze everything except FC layers initially
def freeze_backbones(model):
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

freeze_backbones(fusion_model)

train_losses, val_losses = [], []

total_epochs = FROZEN_EPOCHS + FINE_TUNE_EPOCHS
for epoch in range(total_epochs):
    if epoch == FROZEN_EPOCHS:
        print("Unfreezing all backbones for fine-tuning...")
        unfreeze_all(fusion_model)
        optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-5)

    # --- Training ---
    fusion_model.train()
    running_loss = 0.0
    for batch in train_loader:
        img_emb = batch["img_emb"].to(device)
        text_emb = batch["text_emb"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = fusion_model(img_emb, text_emb)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # --- Validation ---
    fusion_model.eval()
    running_val_loss = 0.0
    correct, total = 0, 0
    misclassified = []
    with torch.no_grad():
        for batch in val_loader:
            img_emb = batch["img_emb"].to(device)
            text_emb = batch["text_emb"].to(device)
            labels = batch["label"].to(device)

            outputs = fusion_model(img_emb, text_emb)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Capture misclassifications
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    misclassified.append(i)

    val_loss = running_val_loss / len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{total_epochs} Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

Saving Files

In [ ]:
# Save best model (or final one)
torch.save(fusion_model.state_dict(), "/home/le.song/Assignment2/fusion_model.pth")

# Loss plot
plt.figure()
plt.plot(range(1, total_epochs+1), train_losses, label="Train Loss")
plt.plot(range(1, total_epochs+1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Fusion Training Loss")
plt.savefig("/home/le.song/Assignment2/fusion_loss_curve.png")
plt.close()

# Misclassified log
with open("/home/le.song/Assignment2/fusion_misclassified.txt", "w") as f:
    for idx in misclassified:
        f.write(f"{idx}\n")  # or map idx to file name/text if you like